In [1]:
import numpy as np 
import pandas as pd 

In [3]:
df = pd.read_csv('train.csv')
df.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


In [5]:
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int) 

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 26 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            699960 non-null  int64  
 1   amount_usd                    699960 non-null  float64
 2   hours_since_last_txn          699960 non-null  float64
 3   txn_count_last_24h            699960 non-null  int64  
 4   distance_from_home_km         699960 non-null  float64
 5   card_age_months               699960 non-null  int64  
 6   customer_age                  699960 non-null  int64  
 7   account_balance_usd           699960 non-null  float64
 8   cvv_retry_count               699960 non-null  int64  
 9   velocity_score                699960 non-null  float64
 10  time_of_day_hour              699960 non-null  int64  
 11  day_of_week                   699960 non-null  int64  
 12  merchant_risk_score           699960 non-null  float64


In [7]:
df.isna().any().any()

np.False_

In [9]:
data = df.drop(columns='is_fraud')
target = df['is_fraud']

data.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,Mastercard,OTP,POS,iPhone,0,0,0,0,0,0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,Visa,PIN,In-App,Android Phone,0,0,0,0,0,0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,Visa,OTP,Online,Tablet,0,0,1,0,0,0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,Mastercard,3D Secure,POS,Android Phone,0,0,0,0,0,0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,Visa,3D Secure,Online,Windows PC,0,0,0,0,0,0


In [10]:
from sklearn.compose import make_column_selector, make_column_transformer 
from sklearn.pipeline import make_pipeline 
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_columns = make_column_selector(dtype_exclude=object)(data)
categorical_columns = make_column_selector(dtype_include= [str, object])(data)

categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")
numerical_preprocessor = StandardScaler() 

preprocessor = make_column_transformer(
    (categorical_preprocessor, categorical_columns),
    (numerical_preprocessor, numerical_columns),
)

In [15]:
from sklearn.linear_model import RidgeClassifier

ridge_clf = make_pipeline(
    preprocessor, 
    RidgeClassifier(random_state=23)
)

In [17]:
from sklearn.model_selection import cross_validate 

def eval_model(clf, data, target):
    cross_val = cross_validate(
        clf, data, target, cv=5, scoring='roc_auc'
    )
    estimator_name = clf[-1].__class__.__name__

    
    return estimator_name, cross_val['test_score'].mean()

In [18]:
eval_model(ridge_clf, data, target)

('RidgeClassifier', np.float64(0.9274366228201781))

In [23]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from catboost import CatBoostClassifier

rfc = make_pipeline(
    preprocessor, 
    RandomForestClassifier(random_state=23, n_jobs=1)
)

gbc = make_pipeline(
    preprocessor, 
    HistGradientBoostingClassifier(random_state=23)
)

cat = make_pipeline(
    preprocessor,
    CatBoostClassifier(random_state=23, verbose=False, thread_count=1)
)

models = [rfc, gbc, cat]

In [30]:
from sklearn.model_selection import train_test_split 
from sklearn.metrics import roc_auc_score

def eval_model_2(model, data, target):
    X_train, X_test, y_train, y_test = train_test_split(
        data,
        target,
        test_size=0.2,
        random_state=23,
        stratify=target,
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "model": model[-1].__class__.__name__,
        "roc_auc": roc_auc_score(y_test, y_prob),
    }


In [32]:
results = [eval_model_2(model, data, target) for model in models]
results

[{'model': 'RandomForestClassifier', 'roc_auc': 0.9081502175841307},
 {'model': 'HistGradientBoostingClassifier', 'roc_auc': 0.939632245216099},
 {'model': 'CatBoostClassifier', 'roc_auc': 0.940461202467157}]

In [35]:
from sklearn.metrics import classification_report

def get_report(model, data, target): 
    X_train, X_test, y_train, y_test = train_test_split(
        data,
        target,
        test_size=0.2,
        random_state=23,
        stratify=target,
    )

    model.fit(X_train, y_train) 
    y_pred = model.predict(X_test) 

    print(classification_report(y_test, y_pred))

In [34]:
[get_report(model, data, target) for model in models]

['              precision    recall  f1-score   support\n\n         0.0       0.99      1.00      0.99    137654\n         1.0       0.86      0.19      0.31      2338\n\n    accuracy                           0.99    139992\n   macro avg       0.92      0.59      0.65    139992\nweighted avg       0.98      0.99      0.98    139992\n',
 '              precision    recall  f1-score   support\n\n         0.0       0.99      1.00      0.99    137654\n         1.0       0.70      0.30      0.42      2338\n\n    accuracy                           0.99    139992\n   macro avg       0.84      0.65      0.71    139992\nweighted avg       0.98      0.99      0.98    139992\n',
 '              precision    recall  f1-score   support\n\n         0.0       0.99      1.00      0.99    137654\n         1.0       0.75      0.32      0.45      2338\n\n    accuracy                           0.99    139992\n   macro avg       0.87      0.66      0.72    139992\nweighted avg       0.98      0.99      0.

In [37]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, recall_score

X_train, X_test, y_train, y_test = train_test_split(
        data,
        target,
        test_size=0.2,
        random_state=23,
        stratify=target,
    ) 

param_grid = {
    "histgradientboostingclassifier__learning_rate": [0.05, 0.1],
    "histgradientboostingclassifier__max_iter": [200, 400],
    "histgradientboostingclassifier__max_leaf_nodes": [15, 31],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=23)

grid = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring=make_scorer(recall_score, pos_label=1),
    cv=cv,
    n_jobs=-1,
    verbose=2,
)

grid.fit(X_train, y_train) 

print(grid.best_params_)
print(grid.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END histgradientboostingclassifier__learning_rate=0.05, histgradientboostingclassifier__max_iter=200, histgradientboostingclassifier__max_leaf_nodes=31; total time=  30.5s
[CV] END histgradientboostingclassifier__learning_rate=0.1, histgradientboostingclassifier__max_iter=200, histgradientboostingclassifier__max_leaf_nodes=31; total time=  12.8s


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END histgradientboostingclassifier__learning_rate=0.05, histgradientboostingclassifier__max_iter=200, histgradientboostingclassifier__max_leaf_nodes=31; total time=  31.2s
[CV] END histgradientboostingclassifier__learning_rate=0.1, histgradientboostingclassifier__max_iter=400, histgradientboostingclassifier__max_leaf_nodes=15; total time=  14.6s
[CV] END histgradientboostingclassifier__learning_rate=0.05, histgradientboostingclassifier__max_iter=400, histgradientboostingclassifier__max_leaf_nodes=15; total time=  35.3s
[CV] END histgradientboostingclassifier__learning_rate=0.1, histgradientboostingclassifier__max_iter=400, histgradientboostingclassifier__max_leaf_nodes=15; total time=  17.0s
[CV] END histgradientboostingclassifier__learning_rate=0.05, histgradientboostingclassifier__max_iter=400, histgradientboostingclassifier__max_leaf_nodes=15; total time=  27.2s
[CV] END histgradientboostingclassifier__learning_rate=0.1, histgradientboostingclassifier__max_iter=200, histgradien

In [38]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, recall_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline

gbc = make_pipeline(
    preprocessor,
    HistGradientBoostingClassifier(
        random_state=23,
        class_weight="balanced",
    ),
)

param_grid = {
    "histgradientboostingclassifier__learning_rate": [0.05, 0.1],
    "histgradientboostingclassifier__max_iter": [200, 400],
    "histgradientboostingclassifier__max_leaf_nodes": [15, 31],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=23)

grid = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    scoring=make_scorer(recall_score, pos_label=1),
    cv=cv,
    n_jobs=-1,
    verbose=2,
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END histgradientboostingclassifier__learning_rate=0.05, histgradientboostingclassifier__max_iter=400, histgradientboostingclassifier__max_leaf_nodes=15; total time=  31.8s
[CV] END histgradientboostingclassifier__learning_rate=0.1, histgradientboostingclassifier__max_iter=200, histgradientboostingclassifier__max_leaf_nodes=31; total time=  16.5s


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


{'histgradientboostingclassifier__learning_rate': 0.05, 'histgradientboostingclassifier__max_iter': 200, 'histgradientboostingclassifier__max_leaf_nodes': 15}
0.8663244572772966


In [39]:
from sklearn.metrics import classification_report

best_gbc = grid.best_estimator_

y_pred = best_gbc.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      0.88      0.93    137654
         1.0       0.11      0.86      0.19      2338

    accuracy                           0.88    139992
   macro avg       0.55      0.87      0.56    139992
weighted avg       0.98      0.88      0.92    139992



In [43]:
from sklearn.metrics import classification_report
import numpy as np

y_prob = best_gbc.predict_proba(X_test)[:, 1]

for threshold in [0.5, 0.7, 0.8, 0.9, 0.93, 0.96, 0.97, 0.98, 0.99]:
    y_pred = (y_prob >= threshold).astype(int)
    print(f"\nThreshold: {threshold}")
    print(classification_report(y_test, y_pred))


Threshold: 0.5
              precision    recall  f1-score   support

         0.0       1.00      0.88      0.93    137654
         1.0       0.11      0.86      0.19      2338

    accuracy                           0.88    139992
   macro avg       0.55      0.87      0.56    139992
weighted avg       0.98      0.88      0.92    139992


Threshold: 0.7
              precision    recall  f1-score   support

         0.0       1.00      0.94      0.97    137654
         1.0       0.17      0.76      0.28      2338

    accuracy                           0.94    139992
   macro avg       0.58      0.85      0.62    139992
weighted avg       0.98      0.94      0.95    139992


Threshold: 0.8
              precision    recall  f1-score   support

         0.0       0.99      0.96      0.98    137654
         1.0       0.24      0.68      0.36      2338

    accuracy                           0.96    139992
   macro avg       0.62      0.82      0.67    139992
weighted avg       0.98   

In [44]:
import numpy as np

for threshold in np.arange(0.80, 0.95, 0.01):
    y_pred = (y_prob >= threshold).astype(int)
    alerts = y_pred.sum()
    print(threshold, alerts)


0.8 6452
0.81 6111
0.8200000000000001 5735
0.8300000000000001 5408
0.8400000000000001 5100
0.8500000000000001 4735
0.8600000000000001 4368
0.8700000000000001 4002
0.8800000000000001 3666
0.8900000000000001 3295
0.9000000000000001 2938
0.9100000000000001 2590
0.9200000000000002 2253
0.9300000000000002 1903
0.9400000000000002 1547


In [45]:

threshold = 0.84  # change this to your chosen capacity limit

y_pred = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.99      0.97      0.98    137654
         1.0       0.29      0.64      0.40      2338

    accuracy                           0.97    139992
   macro avg       0.64      0.80      0.69    139992
weighted avg       0.98      0.97      0.97    139992



In [46]:
# Fit CatBoost on the same training split
cat.fit(X_train, y_train)

# Probability of the positive class
y_prob_cat = cat.predict_proba(X_test)[:, 1]

# Find the threshold that yields about 5,100 alerts
target_alerts = 5100

# Sort probabilities from highest to lowest
order = np.argsort(y_prob_cat)[::-1]
threshold_cat = y_prob_cat[order[target_alerts - 1]]

y_pred_cat = (y_prob_cat >= threshold_cat).astype(int)

print("CatBoost threshold:", threshold_cat)
print("Alerts:", y_pred_cat.sum())
print(classification_report(y_test, y_pred_cat))

CatBoost threshold: 0.0804735784976754
Alerts: 5100
              precision    recall  f1-score   support

         0.0       0.99      0.97      0.98    137654
         1.0       0.29      0.64      0.40      2338

    accuracy                           0.97    139992
   macro avg       0.64      0.81      0.69    139992
weighted avg       0.98      0.97      0.97    139992



In [47]:
test_data = pd.read_csv('test.csv')

# Predict probability of fraud (positive class)
test_prob = best_gbc.predict_proba(test_data)[:, 1]

submission = pd.DataFrame({
    "id": test_data["id"],
    "is_fraud": test_prob,
})

#submission.to_csv("submission.csv", index=False)